# Project Delay Intelligence Dashboard

Open-source Streamlit dashboard for portfolio delay analysis, new-project delay prediction, and SHAP explainability.

In [ ]:
%pip install -U pandas numpy plotly streamlit catboost shap scikit-learn

In [ ]:
import pandas as pd
DATA_PATH = "project_dataset.csv"
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.dtypes)

In [ ]:
TARGET="delay_flag"
LEAKAGE_COLUMNS=["actual_start_date","actual_end_date","actual_duration_days","delay_days","delay_flag","project_status"]
MODEL_FEATURES=["contract_value","planned_duration_days","project_type","location","client_type","project_complexity"]
CATEGORICAL=["project_type","location","client_type","project_complexity"]

print("Historical delay rate:", f"{df[TARGET].mean()*100:.1f}%")
print("Excluded from new-project model:", LEAKAGE_COLUMNS)

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

X=df[MODEL_FEATURES].copy()
y=df[TARGET].astype(int)
for c in CATEGORICAL: X[c]=X[c].fillna("UNKNOWN").astype(str)

cv=StratifiedKFold(5,shuffle=True,random_state=42)
scores=[]
for tr,te in cv.split(X,y):
    m=CatBoostClassifier(iterations=300,depth=4,learning_rate=.04,l2_leaf_reg=5,
        loss_function="Logloss",eval_metric="AUC",random_seed=42,verbose=False)
    m.fit(X.iloc[tr],y.iloc[tr],cat_features=CATEGORICAL)
    scores.append(roc_auc_score(y.iloc[te],m.predict_proba(X.iloc[te])[:,1]))
print("Fold AUC:",[round(s,3) for s in scores])
print("Mean CV AUC:",round(float(np.mean(scores)),3))

In [ ]:
model=CatBoostClassifier(iterations=300,depth=4,learning_rate=.04,l2_leaf_reg=5,
    loss_function="Logloss",eval_metric="AUC",random_seed=42,verbose=False)
model.fit(X,y,cat_features=CATEGORICAL)

import plotly.express as px
imp=pd.DataFrame({"Feature":MODEL_FEATURES,"Importance":model.get_feature_importance()}).sort_values("Importance",ascending=False)
px.bar(imp.sort_values("Importance"),x="Importance",y="Feature",orientation="h",title="Global Feature Importance").show()

In [ ]:
# Example new project
new_project=pd.DataFrame([{
    "contract_value":float(df.contract_value.median()),
    "planned_duration_days":int(df.planned_duration_days.median()),
    "project_type":str(df.project_type.mode()[0]),
    "location":str(df.location.mode()[0]),
    "client_type":str(df.client_type.mode()[0]),
    "project_complexity":str(df.project_complexity.mode()[0])
}])
prob=float(model.predict_proba(new_project[MODEL_FEATURES])[:,1][0])
print(f"Predicted delay probability: {prob*100:.1f}%")

In [ ]:
APP_CODE = 'import warnings\nwarnings.filterwarnings("ignore")\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport plotly.express as px\nimport streamlit as st\nfrom catboost import CatBoostClassifier\nimport shap\n\nst.set_page_config(page_title="Project Delay Intelligence", page_icon="📊", layout="wide")\n\nTARGET = "delay_flag"\nMODEL_FEATURES = ["contract_value","planned_duration_days","project_type","location","client_type","project_complexity"]\nCATEGORICAL = ["project_type","location","client_type","project_complexity"]\n\n@st.cache_data\ndef load_data(uploaded):\n    if uploaded is not None:\n        return pd.read_csv(uploaded)\n    p = Path("project_dataset.csv")\n    return pd.read_csv(p) if p.exists() else pd.DataFrame()\n\ndef prepare_X(d):\n    x = d[MODEL_FEATURES].copy()\n    for c in CATEGORICAL:\n        x[c] = x[c].fillna("UNKNOWN").astype(str)\n    return x\n\n@st.cache_resource\ndef fit_model(d):\n    x, y = prepare_X(d), d[TARGET].astype(int)\n    m = CatBoostClassifier(iterations=300, depth=4, learning_rate=.04,\n        l2_leaf_reg=5, loss_function="Logloss", eval_metric="AUC",\n        random_seed=42, verbose=False)\n    m.fit(x, y, cat_features=CATEGORICAL)\n    return m\n\ndef money(x): return f"{x:,.0f}"\n\nst.title("📊 Project Delay Intelligence Dashboard")\nst.caption("Portfolio analysis • New-project risk prediction • Explainable AI")\n\nuploaded = st.sidebar.file_uploader("Upload project CSV", type=["csv"])\ndf = load_data(uploaded)\nif df.empty:\n    st.info("Upload the CSV or place it beside this app as project_dataset.csv.")\n    st.stop()\n\nneeded = set(MODEL_FEATURES + [TARGET])\nmissing = needed - set(df.columns)\nif missing:\n    st.error(f"Missing required columns: {sorted(missing)}")\n    st.stop()\n\ndf[TARGET] = pd.to_numeric(df[TARGET], errors="coerce").fillna(0).astype(int)\nfor c in ["planned_start_date","planned_end_date","actual_start_date","actual_end_date"]:\n    if c in df: df[c] = pd.to_datetime(df[c], errors="coerce")\n\nst.sidebar.header("Portfolio Filters")\ndef filt(label, col):\n    vals = sorted(df[col].dropna().astype(str).unique())\n    return st.sidebar.multiselect(label, vals, vals)\npt = filt("Project Type","project_type")\ncx = filt("Complexity","project_complexity")\ncl = filt("Client Type","client_type")\nf = df[df.project_type.astype(str).isin(pt) & df.project_complexity.astype(str).isin(cx) & df.client_type.astype(str).isin(cl)].copy()\n\na,b,c,d,e = st.columns(5)\na.metric("Projects", f"{len(f):,}")\nb.metric("Delay Rate", f"{f[TARGET].mean()*100:.1f}%")\nc.metric("Delayed", f"{f[TARGET].sum():,}")\nd.metric("Avg Delay Days", f"{f.loc[f[TARGET].eq(1),\'delay_days\'].mean():.1f}" if "delay_days" in f else "N/A")\ne.metric("Portfolio Value", money(f.contract_value.sum()))\n\ntab1, tab2, tab3 = st.tabs(["📈 Portfolio Analysis","🚨 New Project Risk","🔎 Explainability"])\n\nwith tab1:\n    l,r=st.columns(2)\n    g=f.groupby("project_complexity")[TARGET].agg(["mean","count"]).reset_index()\n    g["delay_rate"]=g["mean"]*100\n    l.plotly_chart(px.bar(g,x="project_complexity",y="delay_rate",text="delay_rate",\n        title="Delay Rate by Complexity",labels={"delay_rate":"Delay Rate (%)"}).update_traces(texttemplate="%{text:.1f}%",textposition="outside"),use_container_width=True)\n    g=f.groupby("project_type")[TARGET].agg(["mean","count"]).reset_index()\n    g["delay_rate"]=g["mean"]*100\n    r.plotly_chart(px.bar(g.sort_values("delay_rate"),x="delay_rate",y="project_type",orientation="h",\n        text="delay_rate",title="Delay Rate by Project Type",labels={"delay_rate":"Delay Rate (%)"}).update_traces(texttemplate="%{text:.1f}%"),use_container_width=True)\n\n    l,r=st.columns(2)\n    r.plotly_chart(px.scatter(f,x="planned_duration_days",y="contract_value",color=TARGET,\n        size="contract_value",hover_data=["project_id","project_type","project_complexity"],\n        log_y=True,title="Contract Value vs Planned Duration"),use_container_width=True)\n    g=f.groupby("location")[TARGET].mean().mul(100).sort_values(ascending=False).head(12).reset_index(name="delay_rate")\n    l.plotly_chart(px.bar(g,x="delay_rate",y="location",orientation="h",text="delay_rate",\n        title="Highest Delay-Rate Locations",labels={"delay_rate":"Delay Rate (%)"}).update_traces(texttemplate="%{text:.1f}%"),use_container_width=True)\n\n    cols=[x for x in ["project_id","project_name","project_type","location","client_type","contract_value",\n                      "planned_duration_days","project_complexity","project_status","delay_days","delay_flag"] if x in f]\n    st.subheader("Project-Level Portfolio")\n    st.dataframe(f[cols].sort_values("delay_flag",ascending=False),use_container_width=True)\n\nwith tab2:\n    st.subheader("New Project Delay Prediction")\n    st.write("Only information available before execution is used by the model.")\n    model=fit_model(df)\n    l,m,r=st.columns(3)\n    ptype=l.selectbox("Project Type",sorted(df.project_type.dropna().astype(str).unique()))\n    loc=l.selectbox("Location",sorted(df.location.dropna().astype(str).unique()))\n    client=l.selectbox("Client Type",sorted(df.client_type.dropna().astype(str).unique()))\n    comp=m.selectbox("Project Complexity",sorted(df.project_complexity.dropna().astype(str).unique()))\n    duration=m.number_input("Planned Duration (days)",1,int(df.planned_duration_days.median()))\n    value=r.number_input("Contract Value",0.0,float(df.contract_value.median()),step=100000.0)\n\n    new=pd.DataFrame([{"contract_value":value,"planned_duration_days":duration,"project_type":ptype,\n                       "location":loc,"client_type":client,"project_complexity":comp}])\n    prob=float(model.predict_proba(new[MODEL_FEATURES])[:,1][0])\n    risk="HIGH" if prob>=.60 else ("MEDIUM" if prob>=.35 else "LOW")\n    x,y,z=st.columns(3)\n    x.metric("Predicted Delay Probability",f"{prob*100:.1f}%")\n    y.metric("Risk Level",risk)\n    z.metric("Historical Delay Rate",f"{df[TARGET].mean()*100:.1f}%")\n    st.progress(prob)\n\n    explainer=shap.TreeExplainer(model)\n    sv=np.asarray(explainer.shap_values(new[MODEL_FEATURES]))[0]\n    ex=pd.DataFrame({"Feature":MODEL_FEATURES,"SHAP Impact":sv})\n    ex["Direction"]=np.where(ex["SHAP Impact"]>0,"Increases delay risk","Reduces delay risk")\n    ex["Absolute Impact"]=ex["SHAP Impact"].abs()\n    ex=ex.sort_values("Absolute Impact",ascending=False)\n    st.plotly_chart(px.bar(ex.sort_values("SHAP Impact"),x="SHAP Impact",y="Feature",\n        color="Direction",orientation="h",title="Why the Model Predicts This Risk"),use_container_width=True)\n    st.dataframe(ex[["Feature","SHAP Impact","Direction"]],use_container_width=True)\n    st.info("SHAP explains model contribution, not causality.")\n\nwith tab3:\n    st.subheader("Global Feature Importance")\n    model=fit_model(df)\n    imp=pd.DataFrame({"Feature":MODEL_FEATURES,"Importance":model.get_feature_importance()}).sort_values("Importance",ascending=False)\n    st.plotly_chart(px.bar(imp.sort_values("Importance"),x="Importance",y="Feature",orientation="h",\n        title="Global Model Feature Importance"),use_container_width=True)\n    st.markdown("""### Modeling governance\n**Included:** contract value, planned duration, project type, location, client type, complexity.\n\n**Excluded:** actual dates, actual duration, realized delay days and project status. These are post-outcome or execution-time fields and would cause target leakage for a new project.\n\n**Caution:** only 150 projects are available. Use this as decision support/prototype evidence, not a production-calibrated probability. Add substantially more historical projects and use chronological validation before deployment.""")\n'
import shap
explainer=shap.TreeExplainer(model)
sv=np.asarray(explainer.shap_values(new_project[MODEL_FEATURES]))[0]
explanation=pd.DataFrame({"Feature":MODEL_FEATURES,"SHAP Impact":sv})
explanation["Direction"]=np.where(explanation["SHAP Impact"]>0,"Increases delay risk","Reduces delay risk")
display(explanation.sort_values("SHAP Impact",ascending=False))
px.bar(explanation.sort_values("SHAP Impact"),x="SHAP Impact",y="Feature",color="Direction",
       orientation="h",title="New Project Risk Explanation").show()

In [ ]:
# Write the Streamlit dashboard
from pathlib import Path
Path("app.py").write_text(APP_CODE, encoding="utf-8")

## Run the dashboard

Save the uploaded CSV as `project_dataset.csv` beside `app.py`, then run:

```bash
pip install -r requirements.txt
streamlit run app.py
```

The dashboard has three tabs: **Portfolio Analysis**, **New Project Risk**, and **Explainability**.